# 🤖 訪問者分類モデルの学習（Google Colab版・改良版）

Google Drive上の画像を使って訪問者分類SVMモデルを学習するノートブックです。

### 改良点
- **データ拡張**: 水平反転・明度変化・軽微な回転で実効データ数を約6倍に増加
- **上半身集中**: 制服が見える上部60%を優先して色特徴を抽出
- **学習/検証分割**: 過学習をチェックする検証セットを確保
- **交差検証**: F1スコアの安定性を確認
- **詳細レポート**: カテゴリ別の精度・混同行列を出力

### 学習カテゴリと制服の特徴
| カテゴリ | フォルダ名 | 制服の色 |
|---------|----------|--------|
| 郵便局（日本郵便） | `postal` | 水色〜青シャツ + 紺ズボン |
| ヤマト運輸 | `yamato` | 深緑 + 黄色ライン |
| 佐川急便 | `sagawa` | 青・白ボーダー + 黒ズボン |
| 住人 | `resident` | 自由（繰り返し来訪者） |
| その他 | `other` | 自由 |

### 手順
1. Google Drive のマイドライブに `LineBot/data/training/` フォルダを作成
2. カテゴリフォルダ（`postal`, `yamato`, `sagawa` など）を作成して画像を配置
3. このノートブックの **「すべてのセルを実行」** を押す
4. 生成された `classifier_svm.pkl` を Raspberry Pi の `LineBot/data/models/` に配置

In [ ]:
# ── 1. Google Drive をマウント ──────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# Google Drive のパス設定（MyDrive の下に LineBot フォルダがある前提）
BASE_DIR      = '/content/drive/MyDrive/LineBot/data'
TRAIN_DIR     = os.path.join(BASE_DIR, 'training')
MODEL_OUT     = os.path.join(BASE_DIR, 'models', 'classifier_svm.pkl')

print('学習データフォルダ:', TRAIN_DIR)
print('モデル保存先      :', MODEL_OUT)

if os.path.exists(TRAIN_DIR):
    categories = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    print('\n検出されたカテゴリ:', categories)
    for cat in sorted(categories):
        imgs = [f for f in os.listdir(os.path.join(TRAIN_DIR, cat)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f'  [{cat}]: {len(imgs)} 枚')
else:
    print(f'❌ フォルダが見つかりません: {TRAIN_DIR}')

In [ ]:
# ── 2. ライブラリのインポートと設定 ──────────────────────────────────────
import cv2
import numpy as np
from glob import glob
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
import joblib
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ─── 設定 ───────────────────────────────────────────────────────────────
IMAGE_SIZE   = (64, 128)  # (幅, 高さ) — HOGDescriptorと合わせる
AUGMENT      = True       # データ拡張を使用する
TEST_SIZE    = 0.2        # 検証データ割合
RANDOM_SEED  = 42

print('✅ ライブラリ読み込み完了')
print(f'   IMAGE_SIZE : {IMAGE_SIZE}')
print(f'   AUGMENT    : {AUGMENT}')
print(f'   TEST_SIZE  : {TEST_SIZE}')

In [ ]:
# ── 3. 特徴量抽出関数とデータ拡張関数の定義 ───────────────────────────────

def extract_features(img: np.ndarray) -> np.ndarray | None:
    """
    BGR画像から 色(HSVヒストグラム) + テクスチャ(HOG) の特徴量を抽出。
    制服が見える上半身(上部60%)を優先する。
    """
    if img is None or img.size == 0:
        return None

    img_resized = cv2.resize(img, IMAGE_SIZE)

    # 上半身（上部60%）を切り出して制服色に集中
    h = img_resized.shape[0]
    upper = img_resized[: int(h * 0.6), :]

    # HSVヒストグラム特徴量（色相16bin, 彩度8bin, 明度8bin）
    hsv = cv2.cvtColor(upper, cv2.COLOR_BGR2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [8],  [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [8],  [0, 256]).flatten()

    # HOG特徴量（輪郭・テクスチャ）
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    hog = cv2.HOGDescriptor(IMAGE_SIZE, (16, 16), (8, 8), (8, 8), 9)
    hog_feat = hog.compute(gray).flatten()

    # 結合してL2正規化
    feat = np.concatenate([hist_h, hist_s, hist_v, hog_feat])
    return feat / (np.linalg.norm(feat) + 1e-9)


def augment_image(img: np.ndarray) -> list:
    """1枚の画像から複数の拡張画像を生成する（元画像含む6枚）。"""
    results = [img]

    # 水平反転
    results.append(cv2.flip(img, 1))

    # 明度 +30
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.int32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] + 30, 0, 255)
    results.append(cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR))

    # 明度 -30
    hsv2 = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.int32)
    hsv2[:, :, 2] = np.clip(hsv2[:, :, 2] - 30, 0, 255)
    results.append(cv2.cvtColor(hsv2.astype(np.uint8), cv2.COLOR_HSV2BGR))

    # 回転 +5°
    center = (img.shape[1] // 2, img.shape[0] // 2)
    M = cv2.getRotationMatrix2D(center, 5, 1.0)
    results.append(cv2.warpAffine(img, M, (img.shape[1], img.shape[0])))

    # 回転 -5°
    M2 = cv2.getRotationMatrix2D(center, -5, 1.0)
    results.append(cv2.warpAffine(img, M2, (img.shape[1], img.shape[0])))

    return results

print('✅ 特徴抽出・拡張関数の定義完了')

In [ ]:
# ── 4. データ読み込み・拡張・特徴量抽出 ─────────────────────────────────

if not os.path.exists(TRAIN_DIR):
    raise FileNotFoundError(f'学習フォルダが見つかりません: {TRAIN_DIR}')

classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
print(f'カテゴリ: {classes}\n')

X, y = [], []
total_original = 0

for label in classes:
    img_paths = (
        glob(os.path.join(TRAIN_DIR, label, '*.jpg'))
        + glob(os.path.join(TRAIN_DIR, label, '*.jpeg'))
        + glob(os.path.join(TRAIN_DIR, label, '*.png'))
    )
    if not img_paths:
        print(f'  [{label}] 画像なし → スキップ')
        continue

    count_before = len(X)
    for path in img_paths:
        img = cv2.imread(path)
        if img is None:
            print(f'    ⚠ 読み込み失敗: {path}')
            continue
        images = augment_image(img) if AUGMENT else [img]
        for aug in images:
            feat = extract_features(aug)
            if feat is not None:
                X.append(feat)
                y.append(label)

    added = len(X) - count_before
    print(f'  [{label}] 元画像: {len(img_paths)}枚 → 拡張後: {added}枚')
    total_original += len(img_paths)

X = np.array(X)
y = np.array(y)
print(f'\n合計: 元画像 {total_original}枚 → 拡張後 {len(X)}枚')
print(f'特徴量次元数: {X.shape[1]}')

In [ ]:
# ── 5. 学習/検証 分割 ────────────────────────────────────────────────────

try:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
except ValueError:
    print('⚠ stratify に失敗（データが少ない）。シンプル分割に切り替え。')
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )

print(f'学習データ: {len(X_train)}件')
print(f'検証データ: {len(X_val)}件')

In [ ]:
# ── 6. SVM 学習 ─────────────────────────────────────────────────────────

print('SVM 学習中...')
clf = SVC(kernel='rbf', probability=True, C=10, gamma='scale', random_state=RANDOM_SEED)
clf.fit(X_train, y_train)
print('✅ 学習完了')

# 交差検証
le = LabelEncoder().fit(y_train)
min_count = min(np.bincount(le.transform(y_train)))
n_folds = min(5, int(min_count))
n_folds = max(2, n_folds)
print(f'\n交差検証 ({n_folds}-fold) 実行中...')
cv_scores = cross_val_score(clf, X_train, y_train, cv=n_folds, scoring='f1_macro')
print(f'CV F1スコア (各fold): {cv_scores.round(3)}')
print(f'CV平均: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# ── 7. 検証データで評価 ──────────────────────────────────────────────────

y_pred = clf.predict(X_val)
label_list = sorted(set(y))

print('=' * 55)
print('検証データ評価結果（未学習データへの汎化性能）')
print('=' * 55)
print(classification_report(y_val, y_pred, zero_division=0))

# 混同行列の可視化
cm = confusion_matrix(y_val, y_pred, labels=label_list)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_list)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix (Validation Set)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()
print('混同行列を /content/confusion_matrix.png に保存しました')

In [ ]:
# ── 8. 全データで再学習してモデルを保存 ─────────────────────────────────

print('全データで最終モデルを学習中...')
clf_final = SVC(kernel='rbf', probability=True, C=10, gamma='scale', random_state=RANDOM_SEED)
clf_final.fit(X, y)

os.makedirs(os.path.dirname(MODEL_OUT), exist_ok=True)
joblib.dump(clf_final, MODEL_OUT)

print(f'\n✅ 学習完了！')
print(f'   保存先: {MODEL_OUT}')
print(f'   学習クラス: {clf_final.classes_}')
print(f'   学習サンプル数: {len(X)}')
print()
print('次のステップ:')
print('  Google Drive から classifier_svm.pkl をダウンロードして')
print('  Raspberry Pi の LineBot/data/models/ フォルダに配置してください。')
print('  main.py を再起動すると自動的にSVMモデルが使われます。')

In [ ]:
# ── 9. サンプル画像の可視化（確認用） ─────────────────────────────────────

fig, axes = plt.subplots(len(classes), 3, figsize=(9, 3 * len(classes)))
if len(classes) == 1:
    axes = [axes]

for row, label in enumerate(classes):
    img_paths = (
        glob(os.path.join(TRAIN_DIR, label, '*.jpg'))
        + glob(os.path.join(TRAIN_DIR, label, '*.png'))
    )[:3]
    for col, path in enumerate(img_paths):
        img = cv2.imread(path)
        if img is not None:
            axes[row][col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row][col].set_title(f'{label}' if col == 0 else '', fontsize=11)
        axes[row][col].axis('off')

plt.suptitle('Training Samples (first 3 per category)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/training_samples.png', dpi=120)
plt.show()
print('サンプル画像を /content/training_samples.png に保存しました')